# Step 4 — Shallow CNN baseline, repeated seeds (IEEE Access revision)

Runtime → Change runtime type → **L4 GPU** (High-RAM if offered). Then **Runtime → Run all**.

Trains the same shallow CNN as the paper on raw and preprocessed PCam and CAMELYON17 (4 arms × 5 seeds = 20 runs).

**Resumable:** finished runs are saved to Drive (`GP_ECG_REVISION/step4_runs`). If Colab disconnects, just Run all again; finished runs are skipped (only the run that was interrupted restarts).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi
!free -g
!df -h /content

In [ ]:
%cd /content
import os
if os.path.isdir('GP_ECG'):
    %cd GP_ECG
    !git pull
else:
    !git clone https://github.com/LeenRayess/reliable-cross-dataset-lnm.git GP_ECG
    %cd GP_ECG
!git log -1 --oneline

### Copy the data from Drive to the fast local disk (~48 GB, about 10–20 min; skipped if already there)

In [ ]:
import shutil
SRC = {
  'pcam_raw': {'train': '/content/drive/MyDrive/GP_ECG_DATA/pcam_raw/training/camelyonpatch_level_2_split_train', 'valid': '/content/drive/MyDrive/GP_ECG_DATA/pcam_raw/val/camelyonpatch_level_2_split_valid', 'test': '/content/drive/MyDrive/GP_ECG_DATA/pcam_raw/test/camelyonpatch_level_2_split_test'},
  'pcam_preprocessed': {s: '/content/drive/MyDrive/GP_ECG_DATA/preprocessed_macenko_benchmark_style/' + s for s in ['train', 'valid', 'test']},
  'cam17_raw': {s: '/content/drive/MyDrive/GP_ECG_DATA/wilds_raw/' + s for s in ['train', 'valid', 'test']},
  'cam17_preprocessed': {s: '/content/drive/MyDrive/GP_ECG_DATA/wilds/preprocessed_macenko_benchmark_style/' + s for s in ['train', 'valid', 'test']},
}
DATA_ROOT = '/content/cnn_data'
for surf, splits in SRC.items():
    os.makedirs(f'{DATA_ROOT}/{surf}', exist_ok=True)
    for split, stem in splits.items():
        for k in ['x', 'y']:
            src, dst = f'{stem}_{k}.h5', f'{DATA_ROOT}/{surf}/{split}_{k}.h5'
            if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(src):
                print('copying', src, flush=True); shutil.copy(src, dst + '.part'); os.replace(dst + '.part', dst)
print('data ready')
for surf in SRC:
    for f in sorted(os.listdir(f'{DATA_ROOT}/{surf}')):
        print(surf, f, round(os.path.getsize(f'{DATA_ROOT}/{surf}/{f}') / 1e9, 2), 'GB')

### Train (about 6–10 h in total; progress prints one line per epoch)

In [ ]:
OUT = '/content/drive/MyDrive/GP_ECG_REVISION'
!python revision/scripts/step4_cnn_seeds.py --data-root /content/cnn_data --arms pcam_raw pcam_preprocessed cam17_preprocessed cam17_raw --seeds 0 1 2 3 4 --out-runs "{OUT}/step4_runs" --out-res "{OUT}/step4_results" --workers 8

### Check: every run should say DONE

In [ ]:
for arm in ['pcam_raw', 'pcam_preprocessed', 'cam17_raw', 'cam17_preprocessed']:
    for s in range(5):
        p = f'{OUT}/step4_runs/{arm}_seed{s}/done.json'
        print(arm, s, 'DONE' if os.path.exists(p) else 'MISSING')